# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a structured guide for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library. All dataset entities (record sets, fields, columns) are referenced by their `@id` in accordance with best practices for handling Croissant datasets.

### Dataset Source
The dataset's Croissant schema is accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and explore the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
List available record sets, their `@id`s, and associated field/column IDs. This provides a map of the dataset entities for subsequent referencing and extraction.

In [ ]:
# List all available record sets and their fields by @id

record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found in the metadata. Please check the schema for data organization.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for idx, rs in enumerate(record_sets):
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', '(no name)')
        print(f"Record Set {idx+1}: @id = {rs_id}, Name = {rs_name}")
        fields = getattr(rs, 'field', [])
        if not fields:
            print("  [No fields defined]")
        else:
            print("  Fields (by @id):")
            for f in fields:
                fid = getattr(f, '@id', None)
                fname = getattr(f, 'name', '(no name)')
                dtype = getattr(f, 'dataType', '(unknown)')
                print(f"    - {fid} (name: {fname}, dataType: {dtype})")

## 3. Data Extraction
Extract data from one or more record sets, using their `@id` for explicit referencing. Data is loaded into pandas DataFrames for inspection and analysis.

In [ ]:
# Build a list of record set @id's for extraction
# In the provided metadata, identify the actual @id's; here we simulate a common pattern from Croissant

# Collect record set @ids
record_set_ids = [getattr(rs, '@id', None) for rs in getattr(metadata, 'recordSet', []) if getattr(rs, '@id', None)]
dataframes = {}

if not record_set_ids:
    print("No record sets discovered for extraction. Please check the Croissant schema content.")
else:
    for record_set_id in record_set_ids:
        print(f"Loading records for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records.")
            print(f"Columns: {df.columns.tolist()}")
        else:
            print(f"No records found for record set {record_set_id}.")
    # Show first few rows for the first record set (if any)
    if dataframes:
        first_id = record_set_ids[0]
        print(f"\nPreview of record set {first_id}:")
        display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and grouping to a numeric field. All fields should be referenced by their `@id` as per best practices.

In [ ]:
# Example: Filter and normalize a numeric field
# Adjust the following variables using field @id's from section 2

# For demonstration, try to detect a reasonable numeric field @id in the first record set
import numpy as np

if not dataframes:
    print("No DataFrames loaded for EDA.")
else:
    record_set_id = list(dataframes.keys())[0]  # Use the first loaded DataFrame
    df = dataframes[record_set_id]
    # Find a candidate numeric field (use first float or integer field if known)
    candidate_numeric_field = None
    for col in df.columns:
        # Heuristic: if values are numeric, treat as candidate
        try:
            if np.issubdtype(df[col].dropna().astype(float).dtype, np.number):
                candidate_numeric_field = col
                break
        except Exception:
            pass
    if not candidate_numeric_field:
        print("No numeric field found to demonstrate filtering.")
    else:
        print(f"Selected numeric field (by @id): {candidate_numeric_field}")
        # Filter for values above an arbitrary threshold (e.g., mean)
        threshold = df[candidate_numeric_field].astype(float).mean()
        filtered_df = df[df[candidate_numeric_field].astype(float) > threshold].copy()
        print(f"Filtered records with {candidate_numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        mean_val = filtered_df[candidate_numeric_field].astype(float).mean()
        std_val = filtered_df[candidate_numeric_field].astype(float).std()
        filtered_df[f"{candidate_numeric_field}_normalized"] = (
            filtered_df[candidate_numeric_field].astype(float) - mean_val) / std_val
        print(f"Normalized {candidate_numeric_field} for filtered records:")
        display(filtered_df[[candidate_numeric_field, f"{candidate_numeric_field}_normalized"]].head())

        # Try to group by a categorical field (use the first non-numeric field if found)
        candidate_group_field = None
        for col in df.columns:
            if col == candidate_numeric_field:
                continue
            if not np.issubdtype(df[col].dropna().astype(str).dtype, np.number):
                candidate_group_field = col
                break
        if candidate_group_field:
            print(f"Grouping by field (by @id): {candidate_group_field}")
            grouped_df = filtered_df.groupby(candidate_group_field).mean(numeric_only=True)
            display(grouped_df.head())
        else:
            print("No categorical field identified for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field using a histogram and, if possible, display boxplots across groups (if any categorical field is found).

In [ ]:
# Simple visualization: histogram and boxplot for numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or not candidate_numeric_field:
    print("Unable to visualize: missing data or numeric field.")
else:
    plt.figure(figsize=(7,4))
    sns.histplot(df[candidate_numeric_field].astype(float), bins=30, kde=True)
    plt.title(f'Distribution of {candidate_numeric_field}')
    plt.xlabel(candidate_numeric_field)
    plt.ylabel('Count')
    plt.show()

    if 'candidate_group_field' in locals() and candidate_group_field:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=df[candidate_group_field], y=df[candidate_numeric_field].astype(float))
        plt.title(f'{candidate_numeric_field} by {candidate_group_field}')
        plt.xlabel(candidate_group_field)
        plt.ylabel(candidate_numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you have:
- Loaded dataset metadata and records using the `mlcroissant` library and referenced entities by their `@id`.
- Explored record sets, field types, and previewed the data structure.
- Performed simple exploratory data analysis with filtering and normalization on a numeric field and grouped records by a categorical field where possible.
- Visualized the numeric field's distribution.

To go further, use field, record set, and column `@id`s for advanced feature engineering or machine learning workflows, following the FAIR principles for reproducible research.